# NB_patient_360

Builds `adb_caresync_health.gold.patient_360` (one row per patient) from the silver layer per the STTM Silver to Gold v1.0. Driving table: `silver.patients`, LEFT JOINed to hospitals, insurance providers and per-patient lab result aggregates. Full rebuild (CREATE OR REPLACE TABLE AS SELECT).

Note: `silver.lab_results` contains exact reload duplicates of `lab_result_id`; they are deduplicated (latest `insert_timestamp`) before aggregation so test counts are not inflated.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync_health.gold;

CREATE OR REPLACE TABLE adb_caresync_health.gold.patient_360 AS
WITH lab_dedup AS (
  SELECT *
  FROM adb_caresync_health.silver.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY insert_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*) AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count,
    MAX(test_date) AS last_test_date
  FROM lab_dedup
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                             AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                         AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                 AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                  AS insurance_provider_id,
  ip.provider_name                                                      AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                              AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                      AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                      AS critical_test_count,
  la.last_test_date
FROM adb_caresync_health.silver.patients p
LEFT JOIN adb_caresync_health.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync_health.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id;

In [0]:

updated_df = spark.table("adb_caresync_health.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))